In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras.utils import to_categorical 

### The Data

Below, the wine dataset is loaded, split, and scaled.  

In [2]:
wine = load_wine(as_frame=True)

In [3]:
wine.frame.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [4]:
wine.frame.info()

<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  target          

In [5]:
X = wine.data
y = to_categorical(wine.target)

In [6]:
X_scaled = StandardScaler().fit_transform(X)

To use the `KerasClassifier`, a function that creates a `keras` model and takes in arguments for the parameters you wish to search. 

In [7]:

tf.random.set_seed(42)
# Function to create a fully connected neural network model for SciKeras
def create_model(optimizer='adam', neurons=50, activation='relu', input_dim=13):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Dense(neurons, activation=activation, input_shape=(input_dim,)))
    model.add(tf.keras.layers.Dense(1))  # Output layer for regression
    model.compile(optimizer=optimizer, loss='mse')
    return model


In [8]:
# Keras model with SciKeras wrapper
model = KerasRegressor(model=create_model, verbose=2)

In [9]:
# Performing the Grid Search
tf.random.set_seed(42)
# Hyperparameters to be optimized
param_grid = {
    'model__neurons': [10, 25, 50, 100],
    'model__activation': ['relu', 'sigmoid'],
    'model__optimizer': ['adam', 'sgd'],
    'batch_size': [1, 10],
    'epochs': [10, 20]
}


In [10]:
# Fit and Evaluate the model

# GridSearchCV for hyperparameter tuning
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3, verbose=2)
grid_result = grid.fit(X_scaled, y)

# Display the best hyperparameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


Fitting 3 folds for each of 64 candidates, totalling 192 fits
Epoch 1/10
118/118 - 1s - 5ms/step - loss: 0.6003
Epoch 2/10
118/118 - 0s - 1ms/step - loss: 0.3773
Epoch 3/10
118/118 - 0s - 1ms/step - loss: 0.3337
Epoch 4/10
118/118 - 0s - 1ms/step - loss: 0.3103
Epoch 5/10
118/118 - 0s - 1ms/step - loss: 0.2949
Epoch 6/10
118/118 - 0s - 1ms/step - loss: 0.2832
Epoch 7/10
118/118 - 0s - 1ms/step - loss: 0.2741
Epoch 8/10
118/118 - 0s - 1ms/step - loss: 0.2671
Epoch 9/10
118/118 - 0s - 1ms/step - loss: 0.2612
Epoch 10/10
118/118 - 0s - 1ms/step - loss: 0.2563
60/60 - 0s - 1ms/step
[CV] END batch_size=1, epochs=10, model__activation=relu, model__neurons=10, model__optimizer=adam; total time=   1.9s
Epoch 1/10
119/119 - 1s - 4ms/step - loss: 0.5592
Epoch 2/10
119/119 - 0s - 1ms/step - loss: 0.3291
Epoch 3/10
119/119 - 0s - 1ms/step - loss: 0.2918
Epoch 4/10
119/119 - 0s - 1ms/step - loss: 0.2748
Epoch 5/10
119/119 - 0s - 1ms/step - loss: 0.2649
Epoch 6/10
119/119 - 0s - 1ms/step - loss: 0.2

In [11]:

# Extract and display results from GridSearchCV
results = pd.DataFrame(grid_result.cv_results_)
print(results.head())

   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       1.732508      0.043249         0.104671        0.005824   
1       1.294715      0.030760         0.098724        0.003627   
2       1.736616      0.053272         0.101237        0.002218   
3       1.377966      0.099694         0.107453        0.005905   
4       1.727865      0.014831         0.099264        0.003000   

   param_batch_size  param_epochs param_model__activation  \
0                 1            10                    relu   
1                 1            10                    relu   
2                 1            10                    relu   
3                 1            10                    relu   
4                 1            10                    relu   

   param_model__neurons param_model__optimizer  \
0                    10                   adam   
1                    10                    sgd   
2                    25                   adam   
3                    25       